# 1. Propósito y alcance

Este notebook construye y verifica formalmente el primer pivote de Schur normalizado. Conserva el orden no conmutativo, separa igualdad exacta de equivalencia módulo compactos y no afirma pertenencia operatorial ni Fredholmness.

## 2. Convenciones de igualdad

- `=` registra una igualdad estructural exacta dentro del cálculo formal y sus hipótesis suministradas.
- `\simeq` registra una equivalencia módulo compactos; no se promueve a igualdad exacta.
- Una sustitución formal y una obligación analítica tienen tipos de traza propios.

In [1]:
from IPython.display import Markdown, Math, display

import symbolic_operator_calculus as soc

trace = soc.build_normalized_first_schur_pivot_derivation()
rendered = soc.render_normalized_first_schur_pivot_latex(trace)
steps = {step.key: step for step in rendered.steps}
[(step.key, step.relation_kind.value) for step in trace.semantic_steps]

[('reduced_definition', 'EXACT_EQUALITY'),
 ('pivot_definition', 'EXACT_EQUALITY'),
 ('normalized_expansion', 'EXACT_EQUALITY'),
 ('inverse_substitution', 'FORMAL_SUBSTITUTION'),
 ('diagonal_recognition', 'EXACT_EQUALITY'),
 ('exact_result', 'EXACT_EQUALITY'),
 ('a21_mod_compact', 'MOD_COMPACT_EQUIVALENCE'),
 ('a12_mod_compact', 'MOD_COMPACT_EQUIVALENCE'),
 ('normalized_mod_compact_result', 'MOD_COMPACT_EQUIVALENCE'),
 ('justify_wh_mod_compact', 'ANALYTIC_PROOF_OBLIGATION'),
 ('classify_complete_correction', 'ANALYTIC_PROOF_OBLIGATION'),
 ('prove_product_closure', 'ANALYTIC_PROOF_OBLIGATION'),
 ('choose_analytic_framework', 'ANALYTIC_PROOF_OBLIGATION'),
 ('apply_fredholm_criterion_later', 'ANALYTIC_PROOF_OBLIGATION')]

## 3. Definición de átomos no conmutativos

In [2]:
atoms = (
    soc.Z1_inverse, soc.Z2_inverse, soc.A12, soc.A21, soc.A22,
    soc.A11_inverse, soc.T1_minus, soc.T2_minus, soc.U1_inverse,
    soc.U2_inverse, soc.P_plus, soc.P_minus, soc.R1, soc.N2,
    soc.N2_first, soc.Vtilde_alpha1, soc.Vtilde_alpha2, soc.G1,
    soc.G2, soc.Wplus_12, soc.Wminus_21,
)
assert all(isinstance(atom, soc.OperatorAtom) for atom in atoms)
assert all(not atom.is_commutative for atom in atoms)
display(Math(r"T_{1,-}=" + soc.render_linear_combination_latex(soc.t1_minus_expression())))
display(Math(r"T_{2,-}=" + soc.render_linear_combination_latex(soc.t2_minus_expression())))
[(atom.name, atom.kind, atom.is_commutative) for atom in atoms]

<IPython.core.display.Math object>

<IPython.core.display.Math object>

[('Z1_inverse', 'multiplication_operator', False),
 ('Z2_inverse', 'multiplication_operator', False),
 ('A12', 'exact_block', False),
 ('A21', 'exact_block', False),
 ('A22', 'exact_block', False),
 ('A11_inverse', 'formal_inverse', False),
 ('T1_minus', 'auxiliary_shift_operator', False),
 ('T2_minus', 'auxiliary_shift_operator', False),
 ('U1_inverse', 'inverse_shift_operator', False),
 ('U2_inverse', 'inverse_shift_operator', False),
 ('P_plus', 'projection', False),
 ('P_minus', 'projection', False),
 ('R1', 'formal_regularizer', False),
 ('N2', 'normalized_pivot', False),
 ('N2_first', 'normalized_schur_pivot', False),
 ('Vtilde_alpha1', 'operator', False),
 ('Vtilde_alpha2', 'operator', False),
 ('G1', 'operator', False),
 ('G2', 'operator', False),
 ('Wplus_12', 'operator', False),
 ('Wminus_21', 'operator', False)]

## 4. Construcción de $A_{2,2}^{(1)}$

In [3]:
assert trace.reduced_definition.right == soc.a22_first_schur_exact_expression()
display(Math(steps["reduced_definition"].latex))

<IPython.core.display.Math object>

## 5. Normalización por $Z_2^{-1}$ y $T_{2,-}$

In [4]:
display(Math(steps["pivot_definition"].latex))
display(Math(steps["normalized_expansion"].latex))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## 6. Sustitución de $A_{1,1}^{(-1)}$

In [5]:
assert trace.semantic_steps[3].relation_kind is soc.DerivationRelationKind.FORMAL_SUBSTITUTION
assert all(
    soc.A11_inverse not in term.product.factors
    for term in trace.inverse_substitution.right.terms
)
display(Math(steps["inverse_rule"].latex))
display(Math(steps["inverse_substitution"].latex))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## 7. Reconocimiento del término $N_2$

In [6]:
display(Math(steps["diagonal_recognition"].latex))

<IPython.core.display.Math object>

## 8. Derivación factorizada de $N_2^{(1)}$

In [7]:
exact_latex = soc.render_normalized_first_schur_pivot_exact_latex(trace)
assert exact_latex == (
    r"N_{2}^{(1)} = N_{2} - Z_{2}^{-1}\,A_{2,1}\,T_{1,-}\,R_{1}\,"
    r"Z_{1}^{-1}\,A_{1,2}\,T_{2,-}"
)
display(Math(exact_latex))

<IPython.core.display.Math object>

## 9. Sustitución módulo compactos de $A_{2,1}$ y $A_{1,2}$

In [8]:
display(Math(steps["offdiagonal_models"].latex))
mod_compact_latex = soc.render_normalized_first_schur_pivot_mod_compact_latex(trace)
assert trace.mod_compact_result.certification_status is soc.CertificationStatus.UNCERTIFIED
display(Math(mod_compact_latex))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## 10. Expansión controlada de las diferencias lineales

In [9]:
expanded_latex = soc.render_normalized_first_schur_pivot_mod_compact_latex(
    trace, expand_differences=True
)
assert expanded_latex.count(r"R_{1}") == 4
assert all(
    term.product.factors.count(soc.R1) == 1
    for term in trace.expanded_mod_compact_result.terms[1:]
)
display(Math(expanded_latex))

<IPython.core.display.Math object>

## 11. Tabla de clasificación de factores

In [10]:
factor_table = soc.render_normalized_factor_classification_markdown(trace)
assert len(trace.factor_classifications) == 11
display(Markdown(factor_table))

| Factor | Clase operatorial conocida | Estatus | Fuente matemática |
|---|---|---|---|
| $Z_{2}^{-1}$ | MultiplicationOperator | exacto | definición del transporte |
| $\widetilde V_{\alpha_2}$ | Multiplication × Dilation | exacto | transporte del shift |
| $G_{2}$ | MultiplicationOperator | exacto | coeficiente branchwise |
| $W^-_{2,1}$ | LocalizedWienerHopfOperator | módulo compactos | localización cuspídea 2025 |
| $T_{1,-}$ | AuxiliaryShiftOperator | invertible | resultado branchwise |
| $R_{1}$ | MellinPDORegularizer | regularizador módulo compactos | KKL 2014, Thm. 5.8 |
| $Z_{1}^{-1}$ | MultiplicationOperator | exacto | normalización diagonal |
| $\widetilde V_{\alpha_1}$ | Multiplication × Dilation | exacto | transporte del shift |
| $G_{1}$ | MultiplicationOperator | exacto | coeficiente branchwise |
| $W^+_{1,2}$ | LocalizedWienerHopfOperator | módulo compactos | localización cuspídea 2025 |
| $T_{2,-}$ | AuxiliaryShiftOperator | invertible | resultado branchwise |

## 12. Obligaciones analíticas pendientes

In [11]:
obligations = soc.render_normalized_proof_obligations_markdown(trace)
assert len(trace.proof_obligations) == 5
assert all(
    item.relation_kind is soc.DerivationRelationKind.ANALYTIC_PROOF_OBLIGATION
    for item in trace.proof_obligations
)
display(Markdown(obligations))

1. Justificar la sustitución Wiener--Hopf módulo compactos.
2. Demostrar que la corrección completa pertenece a una álgebra cerrada apropiada.
3. Demostrar cierre bajo los productos que aparecen.
4. Decidir si la salida es un Mellin PDO admisible o un elemento de la álgebra cuspídea de 2025.
5. Aplicar después un criterio de Fredholmness, sin afirmar aún su conclusión.

## 13. Frontera de capacidad

In [12]:
capability_boundary = """
**La paquetería demuestra estructuralmente:** composición y sustitución formal finitas, orden de factores, signos, reconocimiento de la regla suministrada para $N_2$ y reconstrucción de la expansión a partir de las dos diferencias.

**La paquetería no demuestra:** compactness del residuo, pertenencia a una álgebra cerrada, cierre de productos, admisibilidad Mellin PDO ni Fredholmness. Las equivalencias Wiener--Hopf siguen sin certificar analíticamente.
"""
assert trace.mod_compact_result.certification_status.value == "uncertified"
display(Markdown(capability_boundary))


**La paquetería demuestra estructuralmente:** composición y sustitución formal finitas, orden de factores, signos, reconocimiento de la regla suministrada para $N_2$ y reconstrucción de la expansión a partir de las dos diferencias.

**La paquetería no demuestra:** compactness del residuo, pertenencia a una álgebra cerrada, cierre de productos, admisibilidad Mellin PDO ni Fredholmness. Las equivalencias Wiener--Hopf siguen sin certificar analíticamente.


## 14. Exportación LaTeX

In [13]:
latex_fragment = "\n\n".join(
    (r"\[" + step.latex + r"\]")
    for step in rendered.steps
)
assert r"\documentclass" not in latex_fragment
print(latex_fragment)

\[A_{2,2}^{(1)} = A_{2,2} - A_{2,1}\,A_{1,1}^{(-1)}\,A_{1,2}\]

\[N_{2}^{(1)} = Z_{2}^{-1}\,A_{2,2}^{(1)}\,T_{2,-}\]

\[Z_{2}^{-1}\,A_{2,2}^{(1)}\,T_{2,-} = Z_{2}^{-1}\,A_{2,2}\,T_{2,-} - Z_{2}^{-1}\,A_{2,1}\,A_{1,1}^{(-1)}\,A_{1,2}\,T_{2,-}\]

\[A_{1,1}^{(-1)} \overset{\mathrm{formal}}{=} T_{1,-}\,R_{1}\,Z_{1}^{-1}\]

\[Z_{2}^{-1}\,A_{2,2}\,T_{2,-} - Z_{2}^{-1}\,A_{2,1}\,A_{1,1}^{(-1)}\,A_{1,2}\,T_{2,-} \overset{\mathrm{formal}}{=} Z_{2}^{-1}\,A_{2,2}\,T_{2,-} - Z_{2}^{-1}\,A_{2,1}\,T_{1,-}\,R_{1}\,Z_{1}^{-1}\,A_{1,2}\,T_{2,-}\]

\[Z_{2}^{-1}\,A_{2,2}\,T_{2,-} = N_{2}\]

\[N_{2}^{(1)} = N_{2} - Z_{2}^{-1}\,A_{2,1}\,T_{1,-}\,R_{1}\,Z_{1}^{-1}\,A_{1,2}\,T_{2,-}\]

\[\begin{aligned}A_{2,1} &\simeq - \left(\widetilde V_{\alpha_2} - G_{2}\right)\,W^-_{2,1} \\ A_{1,2} &\simeq \left(\widetilde V_{\alpha_1} - G_{1}\right)\,W^+_{1,2}\end{aligned}\]

\[N_{2}^{(1)} \simeq N_{2} + Z_{2}^{-1}\,\left(\widetilde V_{\alpha_2} - G_{2}\right)\,W^-_{2,1}\,T_{1,-}\,R_{1}\,Z_{1}^{-1}\,\left(\widetilde V_{